# Task 2 - End-to-End Pipeline

This notebook orchestrates the full Task 2 workflow using the reusable logic in `src/connect4`: training, evaluation, PDF export, analysis summary, and representative matches.

By default it runs a smoke configuration so you can execute all cells quickly. Change `RUN_PROFILE` to `"full"` in the config cell when you want the full delivery-sized run.

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, clear_output, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from connect4 import (
    Connect4,
    DEFAULT_TASK2_CONFIG,
    create_board_figure,
    create_task2_results_figure,
    extract_representative_matches,
    replay_match_states,
    run_task2_pipeline,
)


## Configuration

In [ ]:
RUN_PROFILE = os.environ.get('TASK2_NOTEBOOK_PROFILE', 'smoke')  # 'smoke' or 'full'
OUTPUT_DIR_ENV = os.environ.get('TASK2_NOTEBOOK_OUTPUT_DIR')
OUTPUT_DIR = Path(OUTPUT_DIR_ENV) if OUTPUT_DIR_ENV else ROOT / 'artifacts' / f'task2_{RUN_PROFILE}'
VISUALS_DIR = OUTPUT_DIR / 'notebook_visuals'
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

if RUN_PROFILE == 'full':
    training_config = None
    evaluation_config = None
else:
    training_config = {
        'episodes': 6,
        'checkpoint_interval': 3,
        'seed': 15,
        'stats_window_size': 4,
        'initial_epsilon': 0.9,
        'epsilon_end': 0.2,
    }
    evaluation_config = {
        'matches_per_condition': 3,
        'seed': 27,
        'td_epsilon': 0.0,
    }

print(f'Run profile: {RUN_PROFILE}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Visual exports: {VISUALS_DIR}')
if RUN_PROFILE == 'full':
    print('Training config:', DEFAULT_TASK2_CONFIG['training'])
    print('Evaluation config:', DEFAULT_TASK2_CONFIG['evaluation'])
else:
    print('Training config:', training_config)
    print('Evaluation config:', evaluation_config)


## Run Pipeline

In [ ]:
result = run_task2_pipeline(
    OUTPUT_DIR,
    training_config=training_config,
    evaluation_config=evaluation_config,
)

display(Markdown('### Artifacts'))
print('Training summary:', OUTPUT_DIR / 'training' / 'training_summary.json')
print('Evaluation summary:', OUTPUT_DIR / 'evaluation' / 'task2_evaluation_summary.json')
print('PDF figure:', result['report']['pdf_path'])
print('Representative matches:', result['analysis']['representative_matches_path'])
print('Analysis summary:', result['analysis']['analysis_summary_path'])


## Training Overview

In [ ]:
episode_logs = result['training']['episode_logs']
episodes = [row['episode'] for row in episode_logs]
rewards = [row['reference_reward'] for row in episode_logs]
td_errors = [row['mean_abs_td_error'] for row in episode_logs]
epsilons = [row['epsilon'] for row in episode_logs]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(episodes, rewards, marker='o')
axes[0].set_title('Reference Reward by Episode')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward')
axes[0].grid(True, alpha=0.3)

axes[1].plot(episodes, td_errors, marker='o', color='#C0392B')
axes[1].set_title('Mean Absolute TD Error')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Mean |TD error|')
axes[1].grid(True, alpha=0.3)

axes[2].plot(episodes, epsilons, marker='o', color='#2E86C1')
axes[2].set_title('Epsilon Schedule')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Epsilon')
axes[2].grid(True, alpha=0.3)

training_plot_path = VISUALS_DIR / 'training_overview.png'
fig.tight_layout()
fig.savefig(training_plot_path, dpi=150)
plt.close(fig)
display(Image(filename=str(training_plot_path)))

print('Training overview image:', training_plot_path)
print('Final epsilon:', result['training']['final_epsilon'])
print('Checkpoints saved:', len(result['training']['checkpoints']))


## Evaluation Summary

In [ ]:
for condition, condition_summary in result['evaluation']['conditions'].items():
    print(f"Condition {condition}: {condition_summary['agent1_label']} vs {condition_summary['agent2_label']}")
    print(f"  Wins:   {condition_summary['wins']}")
    print(f"  Losses: {condition_summary['losses']}")
    print(f"  Draws:  {condition_summary['draws']}")
    print(f"  Matches:{condition_summary['num_matches']}")
    print()


## Task 2 Figure

In [ ]:
fig, counts = create_task2_results_figure(result['evaluation'])
figure_png_path = VISUALS_DIR / 'task2_results.png'
fig.savefig(figure_png_path, dpi=150)
plt.close(fig)
display(Image(filename=str(figure_png_path)))
print('PNG figure:', figure_png_path)
print('PDF exported to:', result['report']['pdf_path'])


## Analysis Summary

In [ ]:
analysis_summary = result['analysis']['analysis_summary']
print(json.dumps(analysis_summary, indent=2))


## Representative Matches

In [ ]:
representative_matches = extract_representative_matches(result['evaluation'])
representative_image_paths = {}

for condition, match in representative_matches.items():
    print(f"Condition {condition}: {match['agent1_label']} vs {match['agent2_label']}")
    print(f"  Starter: {match['starter_label']}")
    print(f"  Winner:  {match['winner_label'] if match['winner_label'] else 'Draw'}")
    print(f"  Moves:   {match['move_count']}")
    frames = replay_match_states(match)
    final_frame = frames[-1]
    fig, _ = create_board_figure(
        Connect4(board=final_frame['board']),
        title=f"Condition {condition} - Final Board",
        last_move=tuple(final_frame['last_move']) if final_frame['last_move'] is not None else None,
    )
    image_path = VISUALS_DIR / f'representative_condition_{condition}.png'
    fig.savefig(image_path, dpi=150)
    plt.close(fig)
    representative_image_paths[condition] = image_path
    display(Image(filename=str(image_path)))


## Exported Visuals

In [ ]:
print('Notebook visual exports:')
for path in sorted(VISUALS_DIR.glob('*')):
    print(' ', path)


## Optional Replay Helper

In [ ]:
def replay_condition(condition, delay=0.35):
    match = representative_matches[condition]
    frames = replay_match_states(match)
    for frame in frames:
        clear_output(wait=True)
        title = f"Condition {condition} | Ply {frame['ply']}"
        fig, _ = create_board_figure(
            Connect4(board=frame['board']),
            title=title,
            last_move=tuple(frame['last_move']) if frame['last_move'] is not None else None,
        )
        display(fig)
        plt.close(fig)
        time.sleep(delay)

print("Use replay_condition('A'), replay_condition('B'), or replay_condition('C') to animate a representative match.")
